# CCM-109 - Tópicos especiais de IA - Deep Learning

## Pipeline de detecção de deepfake

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtlimo/ccm-109/blob/main/prediction.ipynb)

---

### Índice

1. [Configuração](#1-configuração)
2. [Dataset FF++](#2-dataset-faceforensics)
3. [Funções auxiliares](#3-funções-auxiliares)
4. [Treino](#4-treino)

## 1.Configuração

In [ ]:
%pip install -q tensorflow scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# @title Imports

import os
import gc
import random
from pathlib import Path
from collections import defaultdict
from glob import glob

import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

import preprocess_img as pi

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

2026-07-30 17:52:08.057251: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.20.0
GPU disponível: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# @title Configurações

CFG = {
    "model_path": "/caminho/seu_modelo.keras",
    "img_size": pi.SIGLIP2_INPUT_SIZE,
    "batch_size": 32,
    "threshold": 0.5,

    "aggregation": "median",
    "trim_percent": 10,

    "confidence": pi.CONFIDENCE_THRESHOLD,      # 0.9
    "min_face_size": pi.MIN_FACE_SIZE,          # 80
    "skip_frames": pi.SKIP_FRAMES,              # 9
    "max_frames": pi.MAX_FRAMES_PER_VIDEO,      # 30
    "max_read_limit": pi.MAX_READ_LIMIT,        # 1000
    "jpg_quality": pi.JPG_QUALITY,              # 95

    "pred_output_dir": "/caminho/frames_predicao",   
    "train_output_dir": "/caminho/frames_treino",    
}

## 2. Funções auxiliares

In [ ]:
def aggregate_mean(preds):
    return float(np.mean(preds))


def aggregate_median(preds):
    return float(np.median(preds))


def aggregate_vote(preds, threshold=0.5):
    return float(np.mean(preds > threshold))


def aggregate_trimmed_mean(preds, trim_percent=10):
    lower = np.percentile(preds, trim_percent)
    upper = np.percentile(preds, 100 - trim_percent)
    trimmed = preds[(preds >= lower) & (preds <= upper)]
    return float(np.mean(trimmed))


def aggregate_video(preds, method="median", threshold=0.5, trim_percent=10):
    if method == "mean":
        return aggregate_mean(preds)
    elif method == "median":
        return aggregate_median(preds)
    elif method == "vote":
        return aggregate_vote(preds, threshold)
    elif method == "trimmed_mean":
        return aggregate_trimmed_mean(preds, trim_percent)
    else:
        raise ValueError(f"Método desconhecido: {method}")


def classify_video(video_score, threshold=0.5):
    return "fake" if video_score > threshold else "real"

# 3. Pré-processamento

In [ ]:
def load_preprocessed_frames(video_name, frames_dir):
    video_dir = Path(frames_dir) / video_name
    npy_files = sorted(video_dir.glob("*_siglip2.npy"))

    if not npy_files:
        return None

    frames = [np.load(f) for f in npy_files]
    return np.stack(frames, axis=0)


def preprocess_video_for_prediction(video_path, output_dir=None):
    output_dir = output_dir or CFG["pred_output_dir"]
    video_path = Path(video_path)
    video_name = video_path.stem

    n_faces = pi.process_video(video_path, output_dir)

    if n_faces == 0:
        print(f"[AVISO] Nenhuma face detectada em {video_path.name}")
        return {"tensor": None, "frame_paths": [], "video_name": video_name}

    tensor = load_preprocessed_frames(video_name, output_dir)

    if tensor is None:
        return {"tensor": None, "frame_paths": [], "video_name": video_name}

    video_dir = Path(output_dir) / video_name
    jpg_files = sorted(video_dir.glob("*.jpg"))

    return {
        "tensor": tensor,
        "frame_paths": [str(p) for p in jpg_files],
        "video_name": video_name,
        "n_frames": len(tensor)
    }



ModuleNotFoundError: No module named 'face_extraction'

## 4. Predição

In [ ]:
def predict_video(model, video_path, output_dir=None, cfg=None):
    cfg = cfg or CFG

    pipeline = preprocess_video_for_prediction(video_path, output_dir)

    if pipeline["tensor"] is None:
        return {
            "video_path": str(video_path),
            "video_name": pipeline["video_name"],
            "error": "Nenhuma face detectada",
            "frame_probs": np.array([]),
            "video_score": None,
            "video_label": None,
            "n_frames": 0
        }

    tensor = pipeline["tensor"]

    frame_probs = model.predict(tensor, batch_size=cfg["batch_size"], verbose=0).flatten()

    video_score = aggregate_video(
        frame_probs,
        method=cfg["aggregation"],
        threshold=cfg["threshold"],
        trim_percent=cfg["trim_percent"]
    )

    video_label = classify_video(video_score, cfg["threshold"])

    return {
        "video_path": str(video_path),
        "video_name": pipeline["video_name"],
        "frame_paths": pipeline["frame_paths"],
        "frame_probs": frame_probs,
        "video_score": video_score,
        "video_label": video_label,
        "n_frames": len(frame_probs)
    }


def predict_videos_batch(model, video_paths, output_base_dir=None, cfg=None):
    cfg = cfg or CFG
    results = []

    for i, vp in enumerate(video_paths):
        print(f"\n[{i+1}/{len(video_paths)}] {Path(vp).name}")

        out_dir = None
        if output_base_dir:
            out_dir = Path(output_base_dir) / Path(vp).stem

        result = predict_video(model, vp, out_dir, cfg)
        results.append(result)

        if result.get("error"):
            print(f"   ⚠️  {result['error']}")
        else:
            print(f"   Score: {result['video_score']:.4f} | Label: {result['video_label'].upper()} | Frames: {result['n_frames']}")

    return results

In [ ]:
# @title Avaliação do modelo

def evaluate_predictions(results, true_labels_dict):
    y_true = []
    y_pred = []
    y_scores = []
    errors = []

    for res in results:
        name = res["video_name"]

        if res.get("error"):
            errors.append({"video": name, "error": res["error"]})
            continue

        if name not in true_labels_dict:
            errors.append({"video": name, "error": "Sem ground truth"})
            continue

        y_true.append(true_labels_dict[name])
        y_pred.append(1 if res["video_label"] == "fake" else 0)
        y_scores.append(res["video_score"])

    metrics = {
        "n_evaluated": len(y_true),
        "n_errors": len(errors),
        "accuracy": accuracy_score(y_true, y_pred) if y_true else None,
        "f1": f1_score(y_true, y_pred) if y_true else None,
    }

    if len(set(y_true)) > 1:
        metrics["auc"] = roc_auc_score(y_true, y_scores)

    return {"metrics": metrics, "errors": errors, "details": list(zip(y_true, y_pred, y_scores))}



In [ ]:
# @title Visualização de resultados

def plot_video_prediction(result, cfg=None, figsize=(14, 4)):
    cfg = cfg or CFG

    if result.get("error"):
        print(f"Erro: {result['error']}")
        return

    probs = result["frame_probs"]
    n = len(probs)

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    colors = ["#dc2626" if p > cfg["threshold"] else "#16a34a" for p in probs]
    axes[0].bar(range(n), probs, color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)
    axes[0].axhline(y=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2, label=f"threshold={cfg['threshold']}")
    axes[0].axhline(y=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5, label=f"{cfg['aggregation']}={result['video_score']:.3f}")
    axes[0].set_xlabel("Frame")
    axes[0].set_ylabel("Probabilidade FAKE")
    axes[0].set_title(f"Predições por Frame | {result['video_label'].upper()}")
    axes[0].legend(fontsize=9)
    axes[0].set_ylim(0, 1)

    axes[1].hist(probs, bins=min(20, n), color="#6366f1", alpha=0.7, edgecolor="white")
    axes[1].axvline(x=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5)
    axes[1].axvline(x=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    axes[1].set_xlabel("Probabilidade FAKE")
    axes[1].set_ylabel("Frequência")
    axes[1].set_title("Distribuição")

    axes[2].plot(range(n), probs, color="#8b5cf6", linewidth=1.5, alpha=0.8)
    axes[2].axhline(y=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    axes[2].axhline(y=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5)
    axes[2].fill_between(range(n), 0, probs, where=(probs > cfg["threshold"]), alpha=0.2, color="#dc2626")
    axes[2].fill_between(range(n), 0, probs, where=(probs <= cfg["threshold"]), alpha=0.2, color="#16a34a")
    axes[2].set_xlabel("Frame (tempo)")
    axes[2].set_ylabel("Probabilidade FAKE")
    axes[2].set_title("Evolução Temporal")
    axes[2].set_ylim(0, 1)

    fig.suptitle(f"{result['video_name']} | Frames: {n}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_aggregation_comparison(probs, threshold=0.5, trim_percent=10):
    methods = ["mean", "median", "vote", "trimmed_mean"]
    scores = [aggregate_video(probs, m, threshold, trim_percent) for m in methods]
    labels = [classify_video(s, threshold) for s in scores]
    colors = ["#dc2626" if l == "fake" else "#16a34a" for l in labels]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(methods, scores, color=colors, alpha=0.85, edgecolor="white", linewidth=2)
    ax.axhline(y=threshold, color="#f59e0b", linestyle="--", linewidth=2)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score Agregado")
    ax.set_title("Comparação de Métodos de Agregação")

    for bar, score, label in zip(bars, scores, labels):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.025,
                f"{score:.3f}\n({label.upper()})", ha="center", va="bottom",
                fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()


def plot_dataset_results(results, cfg=None, figsize=(12, 6)):
    cfg = cfg or CFG
    valid = [r for r in results if not r.get("error")]
    if not valid:
        print("Nenhum resultado válido.")
        return

    names = [r["video_name"][:22] for r in valid]
    scores = [r["video_score"] for r in valid]
    labels = [r["video_label"] for r in valid]
    colors = ["#dc2626" if l == "fake" else "#16a34a" for l in labels]

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(range(len(names)), scores, color=colors, alpha=0.85, edgecolor="white")
    ax.axvline(x=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel("Score Agregado")
    ax.set_title(f"Resultados por Vídeo | Agregação: {cfg['aggregation']}")
    ax.set_xlim(0, 1)
    ax.invert_yaxis()

    for bar, score in zip(bars, scores):
        ax.text(score + 0.02, bar.get_y() + bar.get_height()/2, f"{score:.3f}",
                va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()

## 5. Execução da pipeline

In [ ]:
# @title Exemplos de uso


# --- 8.1 TREINO: processar dataset FF++ ---
# stats = process_dataset(
#     dataset_root="/caminho/ff++",
#     output_dir=CFG["train_output_dir"],
#     max_videos_per_folder=100,
#     seed=42
# )

# --- 8.2 PREDIÇÃO: um único vídeo novo ---
# model = keras.models.load_model(CFG["model_path"])
# result = predict_video(model, "/caminho/video_teste.mp4")
# print(f"Score: {result['video_score']:.4f} | Label: {result['video_label']}")
# plot_video_prediction(result)

# --- 8.3 PREDIÇÃO: comparar métodos de agregação ---
# plot_aggregation_comparison(result["frame_probs"])

# --- 8.4 PREDIÇÃO: lote de vídeos ---
# video_list = glob("/caminho/testes/*.mp4")
# results = predict_videos_batch(model, video_list, output_base_dir="/caminho/frames_pred")
# plot_dataset_results(results)

# --- 8.5 AVALIAÇÃO com ground truth ---
# gt = {"video_001": 1, "video_002": 0, "video_003": 1}
# eval_res = evaluate_predictions(results, gt)
# print(f"Acurácia: {eval_res['metrics']['accuracy']:.4f}")
# print(f"F1: {eval_res['metrics']['f1']:.4f}")
